In [1]:
# Set up packages for web scraping
import pandas as pd
import requests
import re
from tqdm.auto import tqdm
import time

In [ ]:
# Display options
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
#pd.reset_option('display.max_rows')
#pd.reset_option('display.max_columns')

In [2]:
# ------------------------------------------------------------------
# Set up query links and headers
# ------------------------------------------------------------------
GRAPHQL_URL = "https://graphql-prod-4877.edge.aws.worldathletics.org/graphql"

HEADERS = {
    "Content-Type": "application/json",
    "x-api-key": "da2-tzmostylynabpfkrgbmmml4toq",
    "User-Agent": "Mozilla/5.0",
}

CATEGORY_QUERY = """
query getRecordsDetailByCategory($categoryId: Int!) {
  getRecordsDetailByCategory(categoryId: $categoryId) {
    gender
    results {
      discipline
      performance
      venue
      date
      progressionListId
      pending
      equal
      womenOnly
      mixed
      yard
      setIndoor
      competitor {
        name
        country
        birthDate
      }
    }
  }
}
"""

PROGRESSION_QUERY = """
query getRecordsDetailByProgression($progressionId: Int!) {
  getRecordsDetailByProgression(progressionId: $progressionId) {
    ageCategory
    environment
    gender
    discipline {
      name
    }
    originType
    origin

    entries {
      progressionListId
      category
      categoryId
      performance
      equal
      pending
      setIndoor
      womenOnly
      mixed
      yard
      wind
      country
      venue
      date

      competitor {
        name
        country
        birthDate
      }
    }
  }
}
"""

In [3]:
# ------------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------------

def infer_measure_type(discipline):

    d = discipline.lower()

    if any(x in d for x in [
        "jump",
        "vault",
        "throw",
        "put",
        "javelin",
        "hammer",
        "discus"
    ]):
        return "distance"

    if any(x in d for x in [
        "decathlon",
        "heptathlon",
        "pentathlon",
        "points"
    ]):
        return "points"

    return "time"


def infer_unit(measure_type):
    return {
        "time": "seconds",
        "distance": "meters",
        "points": "points"
    }[measure_type]


def parse_time(x):
    """
    Convert time strings to total seconds:
        9.58    -> 9.58
        52.3h   -> 52.3
        1:42.31 -> 102.31
        2:00:35 -> 7235
    """
    if pd.isna(x):
        return None

    x = str(x).strip()

    # Remove non-numeric flags like 'h' (hand-timed), 'A' (altitude), '*', etc.
    x = re.sub(r'\([^)]*\)', '', x)
    x = re.sub(r'[A-Za-z*+]', '', x)
    x = x.strip()

    if not x:
        return None

    if ":" not in x:
        return float(x)

    parts = list(map(float, x.split(":")))

    if len(parts) == 2:
        minutes, seconds = parts
        return minutes * 60 + seconds

    if len(parts) == 3:
        hours, minutes, seconds = parts
        return hours * 3600 + minutes * 60 + seconds

    return None


def parse_numeric(x):

    if pd.isna(x):
        return None

    m = re.search(r'-?\d+(?:\.\d+)?', str(x))

    if m:
        return float(m.group())

    return None


def standardize_mark(performance, measure_type, yard=False):
    if performance is None:
        return None

    if measure_type == "time":
        return parse_time(performance)
    
    value = parse_numeric(performance)

    if value is None:
        return None

    if measure_type == "distance" and yard:
        value *= 0.9144

    return value


def build_record_flags(record, indoor=False):

    flags = []

    if record["pending"]:
        flags.append("pending")

    if record["mixed"]:
        flags.append("mixed")

    if record["womenOnly"]:
        flags.append("women_only")

    if indoor:  # Uses the derived indoor flag from venue
        flags.append("indoor")

    if record["yard"]:
        flags.append("yard")

    if record["equal"]:
        flags.append("equal")

    return flags

In [4]:
# ------------------------------------------------------------------
# Records Detail by Category function
# ------------------------------------------------------------------
def get_progression_links(category_id=1):

    payload = {
        "operationName": "getRecordsDetailByCategory",
        "variables": {
            "categoryId": category_id
        },
        "query": CATEGORY_QUERY
    }

    r = requests.post(
        GRAPHQL_URL,
        headers=HEADERS,
        json=payload,
        timeout=30
    )

    r.raise_for_status()

    data = r.json()["data"]["getRecordsDetailByCategory"]

    rows = []

    for gender_group in data:

        gender = gender_group["gender"]

        for rec in gender_group["results"]:

            competitor = rec.get("competitor") or {}

            measure_type = infer_measure_type(rec["discipline"])

            unit = infer_unit(measure_type)

            mark = standardize_mark(
                rec["performance"],
                measure_type,
                yard=rec["yard"]
            )

            # Venue & Indoor parsing
            raw_venue = rec.get("venue") or ""
            indoor = "(i)" in raw_venue.lower()
            clean_venue = re.sub(r'\s*\(i\)', '', raw_venue, flags=re.IGNORECASE).strip()

            # Null-safe check for progressionListId
            raw_prog_id = rec.get("progressionListId")
            if raw_prog_id is not None:
                event_id = raw_prog_id
                progression_url = f"https://worldathletics.org/records/by-progression/{event_id}?type=1"
            else:
                event_id = None
                progression_url = None

            rows.append({

                "gender": gender,
                "discipline": rec["discipline"],
                "measure_type": measure_type,
                "unit": unit,
                "performance_raw": rec["performance"],
                "mark": mark,
                "athlete": competitor.get("name"),
                "country": competitor.get("country"),
                "dob": pd.to_datetime(
                    competitor.get("birthDate"),
                    errors="coerce"
                ),
                "venue": clean_venue,
                "date": pd.to_datetime(
                    rec["date"],
                    errors="coerce"
                ),
                "pending": rec["pending"],
                "mixed": rec["mixed"],
                "women_only": rec["womenOnly"],
                "indoor": indoor,
                "yard": rec["yard"],
                "equal": rec["equal"],
                "record_flags": build_record_flags(rec, indoor=indoor),
                "event_id": event_id,
                "progression_url": progression_url
            })

    # Construct DataFrame and apply nullable integer type conversion
    df = pd.DataFrame(rows)
    df['event_id'] = df['event_id'].astype('Int64')

    return df

In [5]:
# Get the base url tables, the table of all world records that contains the progression sub-links for each event
# Fetch Senior World Records (category 1)
df_wr = get_progression_links(category_id=1)

# Fetch World U20 Records (category 2)
df_u20 = get_progression_links(category_id=2)

# Combine them into a single DataFrame
records_by_category = pd.concat([df_wr, df_u20], ignore_index=True)

In [6]:
# ------------------------------------------------------------------
# Records Detail by Progression function
# ------------------------------------------------------------------
def scrape_progression(event_id):

    payload = {
        "operationName": "getRecordsDetailByProgression",
        "variables": {
            "progressionId": int(event_id)
        },
        "query": PROGRESSION_QUERY
    }

    r = requests.post(
        GRAPHQL_URL,
        headers=HEADERS,
        json=payload,
        timeout=30
    )

    r.raise_for_status()

    result = r.json()["data"]["getRecordsDetailByProgression"]

    discipline = result["discipline"]["name"]
    gender = result["gender"]
    age_category = result["ageCategory"]
    environment = result["environment"]
    origin = result["origin"]
    origin_type = result["originType"]

    measure_type = infer_measure_type(discipline)
    unit = infer_unit(measure_type)

    rows = []

    for i, rec in enumerate(result["entries"], start=1):

        competitor = rec.get("competitor") or {}

        raw_venue = rec.get("venue") or ""
        indoor = rec.get("setIndoor", False) or "(i)" in raw_venue.lower()

        clean_venue = re.sub(
            r"\s*\(i\)",
            "",
            raw_venue,
            flags=re.IGNORECASE
        ).strip()

        rows.append({

            "event_id": int(event_id),
            "record_number": i,
            "discipline": discipline,
            "gender": gender,
            "age_category": age_category,
            "environment": environment,
            "origin": origin,
            "origin_type": origin_type,
            "measure_type": measure_type,
            "unit": unit,
            "performance_raw": rec["performance"],
            "mark": standardize_mark(
                rec["performance"],
                measure_type,
                yard=rec["yard"]
            ),
            "wind": rec["wind"],
            "athlete": competitor.get("name"),
            "country": competitor.get("country"),
            "dob": pd.to_datetime(
                competitor.get("birthDate"),
                errors="coerce"
            ),
            "venue": clean_venue,
            "date": pd.to_datetime(
                rec["date"],
                errors="coerce"
            ),
            "pending": rec["pending"],
            "mixed": rec["mixed"],
            "women_only": rec["womenOnly"],
            "indoor": indoor,
            "yard": rec["yard"],
            "equal": rec["equal"],
            "record_flags": build_record_flags(
                rec,
                indoor=indoor
            ),
            "source_url":
                f"https://worldathletics.org/records/by-progression/{event_id}?type=1"
        })

    return pd.DataFrame(rows)

In [7]:
# Scrape progression tables
all_progressions = []

for event_id in tqdm(
    records_by_category["event_id"].dropna().unique()
):

    try:
        df = scrape_progression(event_id)
        
        if not df.empty:
            all_progressions.append(df)

        time.sleep(0.15)

    except Exception as e:
        print(f"{event_id}: {e}")

progressions = pd.concat(
    all_progressions,
    ignore_index=True
)

progressions = progressions.sort_values(
    ["event_id", "date", "record_number"]
)

progressions["improvement"] = (
    progressions
    .groupby("event_id")["mark"]
    .diff()
)

# For time-based events, a lower mark is better, so diff() will be negative when a record improves. 
# For distance and points, a higher mark is better, so diff() will be positive. 
# Rather than forcing a common sign convention immediately, I'd keep improvement as the raw difference and add a second standardized metric later 
#   (e.g., relative_improvement_pct or performance_gain) once you've assembled the full dataset. 
# That preserves the original measurements while giving you flexibility in how you define "trackflation."

  0%|          | 0/206 [00:00<?, ?it/s]

/var/folders/h4/rl7821mj5377sc5xjcmvrr9w0000gn/T/ipykernel_5362/2683594143.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  progressions = pd.concat(


In [ ]:
# Write the progressions dataset out
progressions.to_csv('/Users/tylerduke/Downloads/NCSU/Blog Post/records_by_progression.csv', index=False)